In [1]:
import sqlite3
import random
import time
import requests
import pandas as pd

random.seed(42)

print("Week 4 environment ready.")

Week 4 environment ready.


In [2]:
# Connect to a local SQLite database
conn = sqlite3.connect("school.db")
cur = conn.cursor()

# Remove old tables if you rerun the notebook
cur.execute("DROP TABLE IF EXISTS enrollments")
cur.execute("DROP TABLE IF EXISTS students")

# Create students table
cur.execute("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    major TEXT,
    gpa REAL
)
""")

# Create enrollments table
cur.execute("""
CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course TEXT,
    grade TEXT,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
)
""")

# Sample student data
students_data = [
    (1, "Alice", "CS", 3.9),
    (2, "Bob", "Business", 3.2),
    (3, "Charlie", "CS", 3.6),
    (4, "Diana", "Engineering", 3.7),
    (5, "Ethan", "CS", 3.4),
    (6, "Fiona", "Business", 3.8),
    (7, "George", "Engineering", 3.1),
    (8, "Hana", "CS", 3.8),
    (9, "Ian", "Business", 3.5),
    (10, "Julia", "Engineering", 3.9)
]

# Sample enrollment data
enrollments_data = [
    (1, 1, "Python", "A"),
    (2, 1, "Databases", "A"),
    (3, 2, "Marketing", "B"),
    (4, 3, "Python", "A"),
    (5, 3, "Statistics", "B+"),
    (6, 4, "Robotics", "A-"),
    (7, 5, "Databases", "B"),
    (8, 6, "Finance", "A"),
    (9, 8, "Python", "A"),
    (10, 9, "Marketing", "B+")
]

cur.executemany(
    "INSERT INTO students VALUES (?, ?, ?, ?)",
    students_data
)

cur.executemany(
    "INSERT INTO enrollments VALUES (?, ?, ?, ?)",
    enrollments_data
)

conn.commit()

print("Database created successfully.")

Database created successfully.


In [3]:
query = """
SELECT *
FROM students
"""

students_df = pd.read_sql(query, conn)

students_df

,student_id,name,major,gpa
0,1,Alice,CS,3.9
1,2,Bob,Business,3.2
2,3,Charlie,CS,3.6
3,4,Diana,Engineering,3.7
4,5,Ethan,CS,3.4
5,6,Fiona,Business,3.8
6,7,George,Engineering,3.1
7,8,Hana,CS,3.8
8,9,Ian,Business,3.5
9,10,Julia,Engineering,3.9


In [5]:
query = """
SELECT name, gpa
FROM students
WHERE major = 'CS' AND gpa >= 3.5
ORDER BY gpa DESC
"""

result = pd.read_sql(query, conn)

result

,name,gpa
0,Alice,3.9
1,Hana,3.8
2,Charlie,3.6


In [6]:
query = """
SELECT
    students.name,
    enrollments.course,
    enrollments.grade
FROM students
INNER JOIN enrollments
    ON students.student_id = enrollments.student_id
"""

join_df = pd.read_sql(query, conn)

join_df

,name,course,grade
0,Alice,Python,A
1,Alice,Databases,A
2,Bob,Marketing,B
3,Charlie,Python,A
4,Charlie,Statistics,B+
5,Diana,Robotics,A-
6,Ethan,Databases,B
7,Fiona,Finance,A
8,Hana,Python,A
9,Ian,Marketing,B+


In [7]:
query = """
SELECT
    students.name,
    enrollments.course,
    enrollments.grade
FROM students
LEFT JOIN enrollments
    ON students.student_id = enrollments.student_id
"""

left_join_df = pd.read_sql(query, conn)

left_join_df

,name,course,grade
0,Alice,Databases,A
1,Alice,Python,A
2,Bob,Marketing,B
3,Charlie,Python,A
4,Charlie,Statistics,B+
5,Diana,Robotics,A-
6,Ethan,Databases,B
7,Fiona,Finance,A
8,George,NaN,NaN
9,Hana,Python,A


In [8]:
query = """
SELECT
    major,
    COUNT(*) AS n_students,
    AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
HAVING COUNT(*) > 2
ORDER BY avg_gpa DESC
"""

grouped_df = pd.read_sql(query, conn)

grouped_df

,major,n_students,avg_gpa
0,CS,4,3.675000
1,Engineering,3,3.566667
2,Business,3,3.500000


In [9]:
major_to_search = "CS"

cur.execute(
    """
    SELECT name, gpa
    FROM students
    WHERE major = ?
    """,
    (major_to_search,)
)

rows = cur.fetchall()

rows

[('Alice', 3.9), ('Charlie', 3.6), ('Ethan', 3.4), ('Hana', 3.8)]

In [10]:
query = """
SELECT
    major,
    AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
"""

sql_df = pd.read_sql(query, conn)

sql_df

,major,avg_gpa
0,Business,3.500000
1,CS,3.675000
2,Engineering,3.566667


In [11]:
conn.close()

print("Database connection closed.")

Database connection closed.


In [12]:
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 37.5665,
    "longitude": 126.9780,
    "current_weather": True,
}

try:
    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    print("Status code:", response.status_code)

    data = response.json()

    print(data["current_weather"])

except requests.exceptions.RequestException:
    data = {
        "current_weather": {
            "temperature": 18.4,
            "windspeed": 9.1
        }
    }

    print("Network request failed. Using sample fallback data.")

Status code: 200
{'time': '2026-10-08T14:45', 'interval': 900, 'temperature': 15.8, 'windspeed': 3.2, 'winddirection': 27, 'is_day': 0, 'weathercode': 0}


In [13]:
weather_df = pd.DataFrame(
    [data["current_weather"]]
)

weather_df

,time,interval,temperature,windspeed,winddirection,is_day,weathercode
0,2026-10-08T14:45,900,15.8,3.2,27,0,0


In [14]:
print("Shape:", weather_df.shape)

print("\nColumns:")
print(weather_df.columns.tolist())

print("\nData types:")
print(weather_df.dtypes)

Shape: (1, 7)

Columns:
['time', 'interval', 'temperature', 'windspeed', 'winddirection', 'is_day', 'weathercode']

Data types:
time                 str
interval           int64
temperature      float64
windspeed        float64
winddirection      int64
is_day             int64
weathercode        int64
dtype: object


In [15]:
if response.status_code == 200:
    print("✅ API request successful.")
else:
    print(
        "⚠ API request returned status:",
        response.status_code
    )

✅ API request successful.


In [16]:
print(data)

{'latitude': 37.55, 'longitude': 127.0, 'generationtime_ms': 0.08046627044677734, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 34.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-10-08T14:45', 'interval': 900, 'temperature': 15.8, 'windspeed': 3.2, 'winddirection': 27, 'is_day': 0, 'weathercode': 0}}


In [17]:
data["current_weather"]

{'time': '2026-10-08T14:45',
 'interval': 900,
 'temperature': 15.8,
 'windspeed': 3.2,
 'winddirection': 27,
 'is_day': 0,
 'weathercode': 0}

In [ ]:
## REST API Notes

- The data was retrieved from the Open-Meteo API.
- The request used the HTTP GET method.
- The response was returned in JSON format.
- The JSON weather data was converted into a Pandas DataFrame.
- HTTP status code 200 indicates a successful request.
- A fallback response can be used if the network request fails.

In [18]:
%pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [3]:
from bs4 import BeautifulSoup

In [5]:
import requests
import time
from bs4 import BeautifulSoup
import pandas as pd

print("Imports loaded successfully.")

Imports loaded successfully.


In [6]:
url = "http://books.toscrape.com"

headers = {
    "User-Agent": "KDU-DataScience-Student-Demo"
}

response = requests.get(
    url,
    headers=headers,
    timeout=10
)

print("Status code:", response.status_code)

Status code: 200


In [7]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import time
from bs4 import BeautifulSoup
import pandas as pd

In [5]:
import requests
import time
from bs4 import BeautifulSoup
import pandas as pd

print("Imports loaded successfully.")

Imports loaded successfully.


In [6]:
url = "http://books.toscrape.com"

headers = {
    "User-Agent": "KDU-DataScience-Student-Demo"
}

response = requests.get(
    url,
    headers=headers,
    timeout=10
)

print("Status code:", response.status_code)

Status code: 200


In [7]:
soup = BeautifulSoup(response.text, "html.parser")

titles = [
    h.get_text(strip=True)
    for h in soup.select("h3 a")
]

print(titles[:5])

['A Light in the ...', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History ...']


In [8]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print(type(soup))

<class 'bs4.BeautifulSoup'>


In [9]:
titles = [
    h.get_text(strip=True)
    for h in soup.select("h3 a")
]

print(titles[:5])

['A Light in the ...', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History ...']


In [10]:
print("Number of titles collected:", len(titles))

Number of titles collected: 20


In [11]:
books_df = pd.DataFrame({
    "title": titles
})

books_df.head()

,title
0,A Light in the ...
1,Tipping the Velvet
2,Soumission
3,Sharp Objects
4,Sapiens: A Brief History ...


In [12]:
time.sleep(1)

print("Paused for 1 second before any next request.")

Paused for 1 second before any next request.


In [ ]:
## Ethical Web Scraping Checklist

Before scraping a website:

1. Check whether an API is available first.
2. Check the site's robots.txt file.
3. Check the Terms of Service.
4. Collect only the data needed for the task.
5. Use a descriptive User-Agent.
6. Add delays between repeated requests.
7. Avoid placing unnecessary load on the website.
8. Store and use the collected data only for a legitimate purpose.

In [13]:
assert response.status_code == 200
assert len(titles) > 0
assert "title" in books_df.columns

print("✅ Web scraping example completed successfully.")

✅ Web scraping example completed successfully.


In [ ]:
## Part 5 — Recap

### What I learned

- Relational databases organize data into linked tables.
- SQL uses `SELECT`, `JOIN`, and `GROUP BY` to retrieve and summarize data.
- `pd.read_sql()` can load SQL query results directly into a Pandas DataFrame.
- REST APIs usually return structured JSON data.
- API responses can be converted into Pandas DataFrames.
- Web scraping should be used only when an API is not available.
- Ethical scraping requires checking robots.txt, Terms of Service, limiting requests, and pacing requests.
- A practical source preference is:

  Database → API → Web Scraping → Pandas

In [14]:
print("=== WEEK 4 LAB CHECK ===")

print("SQL DataFrame available:", "sql_df" in globals())
print("Weather DataFrame available:", "weather_df" in globals())
print("Books DataFrame available:", "books_df" in globals())

print("\n✅ Week 4 tutorial completed.")

=== WEEK 4 LAB CHECK ===
SQL DataFrame available: False
Weather DataFrame available: False
Books DataFrame available: True

✅ Week 4 tutorial completed.
